In [1]:
import os
import glob
import pandas as pd
import pickle
import matplotlib.pyplot as plt
import numpy as np
import random
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta
import pprint
import pyspark
import pyspark.sql.functions as F

from pyspark.sql.functions import col, to_date
from pyspark.sql.types import StringType, IntegerType, FloatType, DateType

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import xgboost as xgb
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import make_scorer, f1_score, roc_auc_score
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

import model_inference


In [2]:
# Build a .py script that takes a snapshot date, loads a model artefact and make an inference and save to datamart

## set up pyspark session

In [3]:
# Initialize SparkSession
spark = pyspark.sql.SparkSession.builder \
    .appName("dev") \
    .master("local[*]") \
    .getOrCreate()

# Set log level to ERROR to hide warnings
spark.sparkContext.setLogLevel("ERROR")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/06/13 07:26:42 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/06/13 07:26:42 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
25/06/13 07:26:42 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


## set up config

In [4]:
snapshot_date_str = "2023-07-01"
model_name = "credit_model_2024_09_01.pkl"


In [5]:
config = {}
config["snapshot_date_str"] = snapshot_date_str
config["snapshot_date"] = datetime.strptime(config["snapshot_date_str"], "%Y-%m-%d")
config["model_name"] = model_name
config["model_bank_directory"] = "model_bank/"
config["model_artefact_filepath"] = config["model_bank_directory"] + config["model_name"]

pprint.pprint(config)

{'model_artefact_filepath': 'model_bank/credit_model_2024_09_01.pkl',
 'model_bank_directory': 'model_bank/',
 'model_name': 'credit_model_2024_09_01.pkl',
 'snapshot_date': datetime.datetime(2023, 7, 1, 0, 0),
 'snapshot_date_str': '2023-07-01'}


## load model artefact from model bank

In [6]:
# Load the model from the pickle file
with open(config["model_artefact_filepath"], 'rb') as file:
    model_artefact = pickle.load(file)

print("Model loaded successfully! " + config["model_artefact_filepath"])

Model loaded successfully! model_bank/credit_model_2024_09_01.pkl


## load feature store

In [7]:
# connect to feature store
folder_path_1 = "datamart/gold/feature_store/eng/"
folder_path_2 = "datamart/gold/feature_store/cust_fin_risk/"
files_list_1 = [folder_path_1+os.path.basename(f) for f in glob.glob(os.path.join(folder_path_1, '*'))]
files_list_2 = [folder_path_2+os.path.basename(f) for f in glob.glob(os.path.join(folder_path_2, '*'))]
feature_store_sdf_1 = spark.read.option("header", "true").parquet(*files_list_1)
feature_store_sdf_2 = spark.read.option("header", "true").parquet(*files_list_2)
features_sdf_1 = feature_store_sdf_1.filter(col("snapshot_date") == config["snapshot_date"])
features_sdf_2 = feature_store_sdf_2
features_sdf_2 = features_sdf_2.drop('snapshot_date')
features_sdf = features_sdf_1.join(features_sdf_2, on=["Customer_ID"], how="left")
features_pdf = features_sdf.toPandas()
features_pdf.info()
features_pdf

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8974 entries, 0 to 8973
Data columns (total 19 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Customer_ID            8974 non-null   object 
 1   snapshot_date          8974 non-null   object 
 2   click_1m               8974 non-null   int32  
 3   click_2m               8974 non-null   int32  
 4   click_3m               8974 non-null   int32  
 5   click_4m               8974 non-null   int32  
 6   click_5m               8974 non-null   int32  
 7   click_6m               8974 non-null   int32  
 8   Credit_History_Age     8974 non-null   int32  
 9   Num_Fin_Pdts           8974 non-null   int32  
 10  EMI_to_Salary          8974 non-null   float64
 11  Debt_to_Salary         8974 non-null   float64
 12  Repayment_Ability      8974 non-null   float64
 13  Loans_per_Credit_Item  8974 non-null   float64
 14  Loan_Extent            8974 non-null   int32  
 15  Outs

,Customer_ID,snapshot_date,click_1m,click_2m,click_3m,click_4m,click_5m,click_6m,Credit_History_Age,Num_Fin_Pdts,EMI_to_Salary,Debt_to_Salary,Repayment_Ability,Loans_per_Credit_Item,Loan_Extent,Outstanding_Debt,Interest_Rate,Delay_from_due_date,Changed_Credit_Limit
0,CUS_0xc65a,2023-07-01,55,0,78,165,53,150,324,5,0.019682,0.339786,2571.387,0.500000,0,891.61,12,0,8.24
1,CUS_0x5e1f,2023-07-01,44,38,290,28,163,104,213,12,0.005982,0.899317,1246.191,0.083333,21,1128.37,9,21,7.24
2,CUS_0x78d3,2023-07-01,114,44,69,59,206,84,318,6,0.020329,0.526027,1359.908,0.400000,10,730.73,2,5,4.57
3,CUS_0x1844,2023-07-01,270,108,202,4,48,44,70,15,15.872294,2.167771,-9485.285,0.230769,36,1382.42,5,12,15.49
4,CUS_0x7f07,2023-07-01,3,0,198,64,241,145,101,14,0.126413,0.730706,371.494,0.500000,45,311.57,8,9,7.91
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8969,CUS_0x8a04,2023-07-01,0,232,157,0,182,176,191,18,0.046804,1.623760,1234.105,0.357143,20,2103.99,15,4,15.30
8970,CUS_0x7f3f,2023-07-01,0,289,0,90,205,240,28,28,0.077371,6.460677,487.555,0.380952,296,3421.09,21,37,23.06
8971,CUS_0x915,2023-07-01,40,72,0,166,260,146,112,9,0.017958,0.415481,1682.422,0.428571,24,712.22,9,8,19.35
8972,CUS_0x4b67,2023-07-01,24,121,1,82,30,124,196,18,0.074758,1.864680,1330.054,0.583333,112,2682.53,28,16,9.44


In [8]:
# rename features
columns_to_exclude = ['Customer_ID', 'snapshot_date']
columns_to_rename = [col for col in features_pdf.columns if col not in columns_to_exclude]
rename_dict = {col: 'feature_' + col for col in columns_to_rename}
features_pdf.rename(columns=rename_dict, inplace=True)
features_pdf.info()
features_pdf

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8974 entries, 0 to 8973
Data columns (total 19 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Customer_ID                    8974 non-null   object 
 1   snapshot_date                  8974 non-null   object 
 2   feature_click_1m               8974 non-null   int32  
 3   feature_click_2m               8974 non-null   int32  
 4   feature_click_3m               8974 non-null   int32  
 5   feature_click_4m               8974 non-null   int32  
 6   feature_click_5m               8974 non-null   int32  
 7   feature_click_6m               8974 non-null   int32  
 8   feature_Credit_History_Age     8974 non-null   int32  
 9   feature_Num_Fin_Pdts           8974 non-null   int32  
 10  feature_EMI_to_Salary          8974 non-null   float64
 11  feature_Debt_to_Salary         8974 non-null   float64
 12  feature_Repayment_Ability      8974 non-null   f

,Customer_ID,snapshot_date,feature_click_1m,feature_click_2m,feature_click_3m,feature_click_4m,feature_click_5m,feature_click_6m,feature_Credit_History_Age,feature_Num_Fin_Pdts,feature_EMI_to_Salary,feature_Debt_to_Salary,feature_Repayment_Ability,feature_Loans_per_Credit_Item,feature_Loan_Extent,feature_Outstanding_Debt,feature_Interest_Rate,feature_Delay_from_due_date,feature_Changed_Credit_Limit
0,CUS_0xc65a,2023-07-01,55,0,78,165,53,150,324,5,0.019682,0.339786,2571.387,0.500000,0,891.61,12,0,8.24
1,CUS_0x5e1f,2023-07-01,44,38,290,28,163,104,213,12,0.005982,0.899317,1246.191,0.083333,21,1128.37,9,21,7.24
2,CUS_0x78d3,2023-07-01,114,44,69,59,206,84,318,6,0.020329,0.526027,1359.908,0.400000,10,730.73,2,5,4.57
3,CUS_0x1844,2023-07-01,270,108,202,4,48,44,70,15,15.872294,2.167771,-9485.285,0.230769,36,1382.42,5,12,15.49
4,CUS_0x7f07,2023-07-01,3,0,198,64,241,145,101,14,0.126413,0.730706,371.494,0.500000,45,311.57,8,9,7.91
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8969,CUS_0x8a04,2023-07-01,0,232,157,0,182,176,191,18,0.046804,1.623760,1234.105,0.357143,20,2103.99,15,4,15.30
8970,CUS_0x7f3f,2023-07-01,0,289,0,90,205,240,28,28,0.077371,6.460677,487.555,0.380952,296,3421.09,21,37,23.06
8971,CUS_0x915,2023-07-01,40,72,0,166,260,146,112,9,0.017958,0.415481,1682.422,0.428571,24,712.22,9,8,19.35
8972,CUS_0x4b67,2023-07-01,24,121,1,82,30,124,196,18,0.074758,1.864680,1330.054,0.583333,112,2682.53,28,16,9.44


## preprocess data for modeling

In [9]:
# prepare X_inference
feature_cols = [fe_col for fe_col in features_pdf.columns if fe_col.startswith('feature_')]
feature_cols
X_inference = features_pdf[feature_cols]

# apply transformer - standard scaler
transformer_stdscaler = model_artefact["preprocessing_transformers"]["stdscaler"]
X_inference = transformer_stdscaler.transform(X_inference)

print('X_inference', X_inference.shape[0])
X_inference

X_inference 8974


array([[-6.30990226e-01, -1.23757966e+00, -3.66269082e-01, ...,
        -3.19442249e-01, -1.40855646e+00, -3.23926195e-01],
       [-7.57310674e-01, -8.06545559e-01,  2.08856031e+00, ...,
        -6.46595410e-01,  1.94481922e-03, -4.75571828e-01],
       [ 4.65467213e-02, -7.38487543e-01, -4.70483537e-01, ...,
        -1.40995279e+00, -1.07272282e+00, -8.80465669e-01],
       ...,
       [-8.03245382e-01, -4.20883469e-01, -1.26946103e+00, ...,
        -6.46595410e-01, -8.71222638e-01,  1.36085679e+00],
       [-9.86984215e-01,  1.34923660e-01, -1.25788164e+00, ...,
         1.42537461e+00, -3.33888818e-01, -1.41951435e-01],
       [ 3.10671294e-01,  6.45358778e-01,  1.47485296e+00, ...,
        -7.55646464e-01, -1.13988955e+00, -2.01093232e-01]],
      shape=(8974, 17))

## model prediction inference

In [10]:
# load model
model = model_artefact["model"]

# predict model
y_inference = model.predict_proba(X_inference)[:, 1]

# prepare output
y_inference_pdf = features_pdf[["Customer_ID","snapshot_date"]].copy()
y_inference_pdf["model_name"] = config["model_name"]
y_inference_pdf["model_predictions"] = y_inference
y_inference_pdf

,Customer_ID,snapshot_date,model_name,model_predictions
0,CUS_0xc65a,2023-07-01,credit_model_2024_09_01.pkl,0.111887
1,CUS_0x5e1f,2023-07-01,credit_model_2024_09_01.pkl,0.172724
2,CUS_0x78d3,2023-07-01,credit_model_2024_09_01.pkl,0.117681
3,CUS_0x1844,2023-07-01,credit_model_2024_09_01.pkl,0.149833
4,CUS_0x7f07,2023-07-01,credit_model_2024_09_01.pkl,0.117825
...,...,...,...,...
8969,CUS_0x8a04,2023-07-01,credit_model_2024_09_01.pkl,0.523286
8970,CUS_0x7f3f,2023-07-01,credit_model_2024_09_01.pkl,0.387615
8971,CUS_0x915,2023-07-01,credit_model_2024_09_01.pkl,0.122115
8972,CUS_0x4b67,2023-07-01,credit_model_2024_09_01.pkl,0.642344


## save model inference to datamart gold table

In [11]:
# create bronze datalake
gold_directory = f"datamart/gold/model_predictions/{config['model_name'][:-4]}/"
print(gold_directory)

if not os.path.exists(gold_directory):
    os.makedirs(gold_directory)

# save gold table - IRL connect to database to write
partition_name = config["model_name"][:-4] + "_predictions_" + config["snapshot_date_str"].replace('-','_') + '.parquet'
filepath = gold_directory + partition_name
spark.createDataFrame(y_inference_pdf).write.mode("overwrite").parquet(filepath)
# df.toPandas().to_parquet(filepath,
#           compression='gzip')
print('saved to:', filepath)

datamart/gold/model_predictions/credit_model_2024_09_01/


saved to: datamart/gold/model_predictions/credit_model_2024_09_01/credit_model_2024_09_01_predictions_2023_07_01.parquet


## backfill

In [12]:
# check date range
snapshot_dates_list = feature_store_sdf_1.select('snapshot_date').distinct().rdd.flatMap(lambda x: x).collect()
snapshot_dates_list.sort()
snapshot_dates_list

['2023-02-01',
 '2023-03-01',
 '2023-04-01',
 '2023-05-01',
 '2023-06-01',
 '2023-07-01',
 '2023-08-01',
 '2023-09-01',
 '2023-10-01',
 '2023-11-01',
 '2023-12-01',
 '2024-01-01',
 '2024-02-01',
 '2024-03-01',
 '2024-04-01',
 '2024-05-01',
 '2024-06-01',
 '2024-07-01',
 '2024-08-01',
 '2024-09-01',
 '2024-10-01',
 '2024-11-01',
 '2024-12-01']

In [13]:
# set up config
snapshot_date_str = "2023-06-01"

start_date_str = "2023-07-01"
end_date_str = "2024-12-01"

In [16]:
# generate list of dates to process
def generate_first_of_month_dates(start_date_str, end_date_str):
    # Convert the date strings to datetime objects
    start_date = datetime.strptime(start_date_str, "%Y-%m-%d")
    end_date = datetime.strptime(end_date_str, "%Y-%m-%d")
    
    # List to store the first of month dates
    first_of_month_dates = []

    # Start from the first of the month of the start_date
    current_date = datetime(start_date.year, start_date.month, 1)

    while current_date <= end_date:
        # Append the date in yyyy-mm-dd format
        first_of_month_dates.append(current_date.strftime("%Y-%m-%d"))
        
        # Move to the first of the next month
        if current_date.month == 12:

            
            current_date = datetime(current_date.year + 1, 1, 1)
        else:
            current_date = datetime(current_date.year, current_date.month + 1, 1)

    return first_of_month_dates

dates_str_lst = generate_first_of_month_dates(start_date_str, end_date_str)


In [17]:
dates_str_lst

['2023-07-01',
 '2023-08-01',
 '2023-09-01',
 '2023-10-01',
 '2023-11-01',
 '2023-12-01',
 '2024-01-01',
 '2024-02-01',
 '2024-03-01',
 '2024-04-01',
 '2024-05-01',
 '2024-06-01',
 '2024-07-01',
 '2024-08-01',
 '2024-09-01',
 '2024-10-01',
 '2024-11-01',
 '2024-12-01']

In [18]:
for snapshot_date in dates_str_lst:
    print(snapshot_date)
    model_inference.main(snapshot_date, model_name)

2023-07-01


---starting job---


{'model_artefact_filepath': 'model_bank/credit_model_2024_09_01.pkl',
 'model_bank_directory': 'model_bank/',
 'model_name': 'credit_model_2024_09_01.pkl',
 'snapshot_date': datetime.datetime(2023, 7, 1, 0, 0),
 'snapshot_date_str': '2023-07-01'}
Model loaded successfully! model_bank/credit_model_2024_09_01.pkl
extracted features_sdf 8974 2023-07-01 00:00:00
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8974 entries, 0 to 8973
Data columns (total 19 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Customer_ID                    8974 non-null   object 
 1   snapshot_date                  8974 non-null   object 
 2   feature_click_1m               8974 non-null   int32  
 3   feature_click_2m               8974 non-null   int32  
 4   feature_click_3m               8974 non-null   int32  
 5   feature_click_4m               8974 non-null   int32  
 6   feature_click

saved to: datamart/gold/model_predictions/credit_model_2024_09_01/credit_model_2024_09_01_predictions_2023_07_01.parquet
2023-08-01


---starting job---


{'model_artefact_filepath': 'model_bank/credit_model_2024_09_01.pkl',
 'model_bank_directory': 'model_bank/',
 'model_name': 'credit_model_2024_09_01.pkl',
 'snapshot_date': datetime.datetime(2023, 8, 1, 0, 0),
 'snapshot_date_str': '2023-08-01'}
Model loaded successfully! model_bank/credit_model_2024_09_01.pkl
extracted features_sdf 8974 2023-08-01 00:00:00
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8974 entries, 0 to 8973
Data columns (total 19 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Customer_ID                    8974 non-null   object 
 1   snapshot_date                  8974 non-null   object 
 2   feature_click_1m               8974 non-null   int32  
 3   feature_click_2m               8974 non-null   int32  
 4   feature_clic

saved to: datamart/gold/model_predictions/credit_model_2024_09_01/credit_model_2024_09_01_predictions_2023_10_01.parquet
2023-11-01


---starting job---


{'model_artefact_filepath': 'model_bank/credit_model_2024_09_01.pkl',
 'model_bank_directory': 'model_bank/',
 'model_name': 'credit_model_2024_09_01.pkl',
 'snapshot_date': datetime.datetime(2023, 11, 1, 0, 0),
 'snapshot_date_str': '2023-11-01'}
Model loaded successfully! model_bank/credit_model_2024_09_01.pkl
extracted features_sdf 8974 2023-11-01 00:00:00
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8974 entries, 0 to 8973
Data columns (total 19 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Customer_ID                    8974 non-null   object 
 1   snapshot_date                  8974 non-null   object 
 2   feature_click_1m               8974 non-null   int32  
 3   feature_click_2m               8974 non-null   int32  
 4   feature_cli

saved to: datamart/gold/model_predictions/credit_model_2024_09_01/credit_model_2024_09_01_predictions_2023_11_01.parquet
2023-12-01


---starting job---


{'model_artefact_filepath': 'model_bank/credit_model_2024_09_01.pkl',
 'model_bank_directory': 'model_bank/',
 'model_name': 'credit_model_2024_09_01.pkl',
 'snapshot_date': datetime.datetime(2023, 12, 1, 0, 0),
 'snapshot_date_str': '2023-12-01'}
Model loaded successfully! model_bank/credit_model_2024_09_01.pkl
extracted features_sdf 8974 2023-12-01 00:00:00
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8974 entries, 0 to 8973
Data columns (total 19 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Customer_ID                    8974 non-null   object 
 1   snapshot_date                  8974 non-null   object 
 2   feature_click_1m               8974 non-null   int32  
 3   feature_click_2m               8974 non-null   int32  
 4   feature_cli

saved to: datamart/gold/model_predictions/credit_model_2024_09_01/credit_model_2024_09_01_predictions_2023_12_01.parquet
2024-01-01


---starting job---


{'model_artefact_filepath': 'model_bank/credit_model_2024_09_01.pkl',
 'model_bank_directory': 'model_bank/',
 'model_name': 'credit_model_2024_09_01.pkl',
 'snapshot_date': datetime.datetime(2024, 1, 1, 0, 0),
 'snapshot_date_str': '2024-01-01'}
Model loaded successfully! model_bank/credit_model_2024_09_01.pkl
extracted features_sdf 8974 2024-01-01 00:00:00
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8974 entries, 0 to 8973
Data columns (total 19 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Customer_ID                    8974 non-null   object 
 1   snapshot_date                  8974 non-null   object 
 2   feature_click_1m               8974 non-null   int32  
 3   feature_click_2m               8974 non-null   int32  
 4   feature_clic

saved to: datamart/gold/model_predictions/credit_model_2024_09_01/credit_model_2024_09_01_predictions_2024_01_01.parquet
2024-02-01


---starting job---


{'model_artefact_filepath': 'model_bank/credit_model_2024_09_01.pkl',
 'model_bank_directory': 'model_bank/',
 'model_name': 'credit_model_2024_09_01.pkl',
 'snapshot_date': datetime.datetime(2024, 2, 1, 0, 0),
 'snapshot_date_str': '2024-02-01'}
Model loaded successfully! model_bank/credit_model_2024_09_01.pkl
extracted features_sdf 8974 2024-02-01 00:00:00
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8974 entries, 0 to 8973
Data columns (total 19 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Customer_ID                    8974 non-null   object 
 1   snapshot_date                  8974 non-null   object 
 2   feature_click_1m               8974 non-null   int32  
 3   feature_click_2m               8974 non-null   int32  
 4   feature_clic

saved to: datamart/gold/model_predictions/credit_model_2024_09_01/credit_model_2024_09_01_predictions_2024_02_01.parquet
2024-03-01


---starting job---


{'model_artefact_filepath': 'model_bank/credit_model_2024_09_01.pkl',
 'model_bank_directory': 'model_bank/',
 'model_name': 'credit_model_2024_09_01.pkl',
 'snapshot_date': datetime.datetime(2024, 3, 1, 0, 0),
 'snapshot_date_str': '2024-03-01'}
Model loaded successfully! model_bank/credit_model_2024_09_01.pkl
extracted features_sdf 8974 2024-03-01 00:00:00
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8974 entries, 0 to 8973
Data columns (total 19 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Customer_ID                    8974 non-null   object 
 1   snapshot_date                  8974 non-null   object 
 2   feature_click_1m               8974 non-null   int32  
 3   feature_click_2m               8974 non-null   int32  
 4   feature_clic

saved to: datamart/gold/model_predictions/credit_model_2024_09_01/credit_model_2024_09_01_predictions_2024_08_01.parquet
2024-09-01


---starting job---


{'model_artefact_filepath': 'model_bank/credit_model_2024_09_01.pkl',
 'model_bank_directory': 'model_bank/',
 'model_name': 'credit_model_2024_09_01.pkl',
 'snapshot_date': datetime.datetime(2024, 9, 1, 0, 0),
 'snapshot_date_str': '2024-09-01'}
Model loaded successfully! model_bank/credit_model_2024_09_01.pkl
extracted features_sdf 8974 2024-09-01 00:00:00
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8974 entries, 0 to 8973
Data columns (total 19 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Customer_ID                    8974 non-null   object 
 1   snapshot_date                  8974 non-null   object 
 2   feature_click_1m               8974 non-null   int32  
 3   feature_click_2m               8974 non-null   int32  
 4   feature_clic

saved to: datamart/gold/model_predictions/credit_model_2024_09_01/credit_model_2024_09_01_predictions_2024_09_01.parquet
2024-10-01


---starting job---


{'model_artefact_filepath': 'model_bank/credit_model_2024_09_01.pkl',
 'model_bank_directory': 'model_bank/',
 'model_name': 'credit_model_2024_09_01.pkl',
 'snapshot_date': datetime.datetime(2024, 10, 1, 0, 0),
 'snapshot_date_str': '2024-10-01'}
Model loaded successfully! model_bank/credit_model_2024_09_01.pkl
extracted features_sdf 8974 2024-10-01 00:00:00
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8974 entries, 0 to 8973
Data columns (total 19 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Customer_ID                    8974 non-null   object 
 1   snapshot_date                  8974 non-null   object 
 2   feature_click_1m               8974 non-null   int32  
 3   feature_click_2m               8974 non-null   int32  
 4   feature_cli

saved to: datamart/gold/model_predictions/credit_model_2024_09_01/credit_model_2024_09_01_predictions_2024_12_01.parquet


## Check datamart

In [19]:
# Initialize SparkSession
spark = pyspark.sql.SparkSession.builder \
    .appName("dev") \
    .master("local[*]") \
    .getOrCreate()

# Set log level to ERROR to hide warnings
spark.sparkContext.setLogLevel("ERROR")

In [20]:
folder_path = "datamart/gold/model_predictions/credit_model_2024_09_01/"
files_list = [folder_path+os.path.basename(f) for f in glob.glob(os.path.join(folder_path, '*'))]
df = spark.read.option("header", "true").parquet(*files_list)
print("row_count:",df.count())

df.show()

row_count: 161532
+-----------+-------------+--------------------+-------------------+
|Customer_ID|snapshot_date|          model_name|  model_predictions|
+-----------+-------------+--------------------+-------------------+
| CUS_0xc5cc|   2023-07-01|credit_model_2024...| 0.3632813530459547|
| CUS_0x5f86|   2023-07-01|credit_model_2024...|0.16103133551597643|
| CUS_0xa788|   2023-07-01|credit_model_2024...| 0.6494344721660852|
| CUS_0xb756|   2023-07-01|credit_model_2024...| 0.3881257110051408|
| CUS_0x8b96|   2023-07-01|credit_model_2024...|0.38867401738137497|
| CUS_0x5a7d|   2023-07-01|credit_model_2024...|0.12037987053539581|
| CUS_0xc653|   2023-07-01|credit_model_2024...|0.11408322283981061|
| CUS_0x8d74|   2023-07-01|credit_model_2024...|0.12556260640182812|
| CUS_0x94f4|   2023-07-01|credit_model_2024...|0.08834903844546463|
| CUS_0x2296|   2023-07-01|credit_model_2024...|0.11764562483731492|
| CUS_0x85f4|   2023-07-01|credit_model_2024...|0.11499523001213881|
| CUS_0x2d84|   